# SROIE OCR Error Analysis

This notebook evaluates whether validated SROIE ground-truth values are recoverable from the existing Tesseract OCR output. It uses substring matching after lightweight evaluation-only normalization; it does not extract fields, alter OCR text, modify raw data, or train a model.

In [14]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.sroie import load_sroie_dataset

DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw'
OCR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sroie_ocr.jsonl'
FIELDS = ['company', 'date', 'address', 'total']

dataset = load_sroie_dataset(DATASET_ROOT, strict=False)
print(f'Validated SROIE samples: {len(dataset)}')
print(f'OCR results path: {OCR_PATH}')
print(f'OCR results available: {OCR_PATH.is_file()}')

Validated SROIE samples: 624
OCR results path: c:\Users\chech\Documents\AI-Document-Intelligence\data\processed\sroie_ocr.jsonl
OCR results available: True


## Load OCR results

The loader below reads each JSONL record without rewriting its `text` value. Joining is by the stable `image_id` emitted by the existing batch OCR utility.

In [8]:
def load_ocr_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with path.open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'OCR line {line_number} is not a JSON object')
            records.append(record)
    return pd.DataFrame.from_records(records)

if OCR_PATH.is_file():
    ocr_results = load_ocr_jsonl(OCR_PATH)
    required_ocr_columns = {'image_id', 'text'}
    missing_ocr_columns = required_ocr_columns - set(ocr_results.columns)
    if missing_ocr_columns:
        raise ValueError(f'OCR JSONL is missing columns: {sorted(missing_ocr_columns)}')
    print(f'OCR records loaded: {len(ocr_results)}')
else:
    ocr_results = pd.DataFrame(columns=['image_id', 'text', 'status'])
    print('No full OCR JSONL found. Run the baseline batch OCR first; analysis tables will be empty.')

ocr_by_id = ocr_results.drop_duplicates('image_id', keep='first').set_index('image_id')
analysis = dataset[['image_id', *FIELDS]].copy()
analysis['ocr_text'] = analysis['image_id'].map(ocr_by_id['text']).fillna('')
if 'status' in ocr_by_id:
    analysis['ocr_status'] = analysis['image_id'].map(ocr_by_id['status']).fillna('missing')
else:
    analysis['ocr_status'] = 'unknown'

assert len(analysis) == len(dataset)
print(f'Joined analysis rows: {len(analysis)}')
display(analysis[['image_id', 'ocr_status']].head())

OCR records loaded: 624
Joined analysis rows: 624


,image_id,ocr_status
0,X00016469612,success
1,X00016469619,success
2,X00016469620,success
3,X00016469622,success
4,X00016469623,success


## Evaluation-only normalization

Normalization removes formatting noise only: case, whitespace/line breaks, and the decimal separator in numeric text. The original ground truth and OCR text remain in `analysis` unchanged. No regex-based field extraction is used.

In [9]:
def normalize_for_match(value: object) -> str:
    text = str(value) if value is not None else ''
    text = text.casefold().replace(',', '.')
    return re.sub(r'\s+', '', text)

def contains_ground_truth(ground_truth: object, ocr_text: object) -> bool:
    normalized_truth = normalize_for_match(ground_truth)
    normalized_ocr = normalize_for_match(ocr_text)
    return bool(normalized_truth) and normalized_truth in normalized_ocr

for field in FIELDS:
    analysis[f'{field}_valid'] = analysis[field].fillna('').astype(str).str.strip().ne('')
    analysis[f'{field}_match'] = [
        contains_ground_truth(truth, ocr) if valid else False
        for truth, ocr, valid in zip(analysis[field], analysis['ocr_text'], analysis[f'{field}_valid'])
    ]

assert analysis['ocr_text'].equals(analysis['image_id'].map(ocr_by_id['text']).fillna(''))
print('Original OCR text preserved: yes')

Original OCR text preserved: yes


## Coverage results

In [10]:
coverage_rows = []
for field in FIELDS:
    valid_count = int(analysis[f'{field}_valid'].sum())
    matched_count = int(analysis[f'{field}_match'].sum())
    coverage_rows.append({
        'field': field,
        'valid_samples': valid_count,
        'normalized_ground_truth_found': matched_count,
        'coverage_percent': round(100 * matched_count / valid_count, 2) if valid_count else 0.0,
    })

coverage = pd.DataFrame(coverage_rows).set_index('field')
display(coverage)

,valid_samples,normalized_ground_truth_found,coverage_percent
field,,,
company,624,327,52.40
date,624,462,74.04
address,624,246,39.42
total,624,478,76.60


## Representative matches and mismatches

Examples show the original OCR text exactly as loaded, including line breaks and OCR errors.

In [11]:
def show_examples(field: str, matched: bool, limit: int = 3) -> None:
    subset = analysis[(analysis[f'{field}_valid']) & (analysis[f'{field}_match'] == matched)]
    columns = ['image_id', field, 'ocr_status', 'ocr_text']
    title = 'successful matches' if matched else 'OCR mismatches'
    print(f'\n{field}: {title} ({len(subset)} available)')
    display(subset[columns].head(limit).style.set_properties(subset=['ocr_text'], **{'white-space': 'pre-wrap'}))

for field in FIELDS:
    show_examples(field, matched=True)
    show_examples(field, matched=False)


company: successful matches (327 available)


,image_id,company,ocr_status,ocr_text
3,X00016469622,YONGFATT ENTERPRISE,success,"tan woon yann YONGFATT ENTERPRISE (4MO517726) NO 122.124. JALAN DEDAP 13 81100 JOHOR BAHRU TEL © 3888, GST ID 000849812504 SIMPLIFIED TAX INVOICE CASH Doc No 0031664 — Date 26/12/2018 USER Time 123100 sperson Ret fom Qty SiPrice Amount Tax £8318 1 8094 8091 SR ELEGANT SCH TR BAG 15 Total Qty 1 8094 Total Sales (Excluding GST) 6091 Discount 000 Total GST 0.00 Rounding 001 elusive of GST) = 80.90 CASH : 100.00 Change: 19.10 | AmtU{RM) Tax (Rm) | 80.91 0.00! Total 80.91 0.00| ARE NOT REFUNDABLE ONLY"
5,X00016469669,ABC HO TRADING,success,"tan chay yee ABC HO TRADING No,2&4, JALAN HARMONT 3/2, TAMAN DESA HARMONT. 81100 JOHOR BAHRUJOHOR 07-355 2616 Cash Bill ; ~01-143008 Date —; 09/01/2019 8:01:11 PM Cashier: OL Description Qty = Price: Amount RM “T — ""2 15.50 Total Amount: Cash Received Change : *Goods Sold are non-refundabl ***Thank You*** 3 ).00 **""Please Come Again***"
6,X00016469672,SOON HUAT MACHINERY ENTERPRISE,success,"tan chay yee SOON HUAT MACHINERY ENTERPRISE (JM0352019-K) NO.53 JALAN PUTRA 1, TAMAN SRI PUTRA, 81200 JOHOR BAHRU JOHOR TEL : 07-5547360 /016-7993391 FAX : 07-5624059 SOONHUAT2000@HOTMAIL.COM GST ID : 002116837376 = _____ CASH SALES _ see ee Doc No. s00004040 Date: 11/01/2019 Cashier USER Time: 09:44:00 Sale! Ref. ‘tem Qty S/Price__S/Price __Amount_Tax 1072 1 80.00 10 80.00 REPAIR ENGINE POWER SPRAYER (1UNIT) workmanship & service 70349 1 160.00 160.00 160.00 GIANT 606 OVERFLOW ASSY 1071 + 17.00 17.00 17.00 ENGINE OIL 70791 1 10.00 1000 10.00 GREASE FOR TOOLS 40ML (AKODA) 70637 1 6.00 6.00 6.00 EY20 PLUG CHAMPION 1643 1 goo 8.00 8.00 STARTER TALI 70197 1 10.00 10.00 10.00 EY20 STARTER HANDLE 70561 2 18.00 18,00 36.00 HD40 1L COTIN Toy = 9 327 00 TotalSales ; 327.00 Discount 0.00 Total 0.00 Rounding 0,00 Total Sales : 327.00 CASH : 327.00 Change : 0.00 GOODS SOLD ARE NOT RETURNABLE, THANK YOU."



company: OCR mismatches (297 available)


,image_id,company,ocr_status,ocr_text
0,X00016469612,BOOK TA .K (TAMAN DAYA) SDN BHD,success,"tan woon yann BOOK TA -K (TAMAN DAYA) SDN BHD PRON 7-W NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU, JOHOR. WAM MITA At Document Ho : TDO1167104 Date 25/12/2018 8:13:39 PM Cashier MANIS. Member CASH BILL CODE/DESC PRICE Dise AMOUIHT Quy RM RM 0556929040118 AF MODELLING CLAY KIDDY FISH 1PC + 9.00) 0,00 9.00 Total : 9,00 Rour ding Adjustment 0.00 Round. :d Total (RM): 9.00 Cash oy 40.00. CHANGE 00 GOODS SOLD ARE NOT RETURNAP EXCHANGEABLE THANK YOU PLEASE COME AGAIN t"
1,X00016469619,INDAH GIFT & HOME DECO,success,"tan woon yann INDAH GIFT & HOME BECO 27, JALAN DEDAP 13, TANAN JOHOR JAYA, 81100 JOHOR BAHRU, JOHOR. Tel:07-3507405 Fax:07-35598160 RECEIPT 19/10/2018 20:49:59 #01 Cashier: CN Location/SP: 05 /0531 MB: 4026588 Room No: Of 050100025279 Desc/Iten Qty Price Ant/RH) ST-PRIVILEGE CARD/GD INDAH 88888 1 10.00 10.00 GF-TABLE LAMP/STITCH 62483 1 55.90 @PIsc 10.00% HTotal Qty 2 TOTAL AHT..+« ROUNDING ADI. CASH... CHANGE. « RH 10.00 Thank You ! Please Come Again ! Goods Sold Are Not Retur Healing In Wholesale And"
2,X00016469620,MR D.I.Y. (JOHOR) SDN BHD,success,"tan woon yann MR O.1.¥. (JOHOR) SDN BHD (CO.REG :933109-x) LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG , 43300 SERI KEMBANGAN, SELANGOR (MR DIY TESCO TERBAU) ~INVOICE~ CHOPPING BOARO 35.5x25.5CM 803M# EZiOHDOS - 24 8970669 1X 19.00 19.00 AIR PRESSURE SPRAYER SX~575-1 1.5L HCO3-7 - 15 9066468 8.02 8.02 WAXCO WINDSHILED cLeaneR” 120ML WAI4-3A - 48 9557031100236 1X 3.02 3.02 BOPP TAPE 48MM#100M CLEAR FZ-04 - 36 6935818350846 1X 3.88 3,88 Item(s) : 4 TOTAL RM 33, 92 ROUNDING ADJUSTMED -RM 0.02 TOTAL ROUNDED RH 33,90 CASH RM 50.00 CHANGE RM 16.10 #2222222 +--+ ---- ed 12-01-19 21:13 SHO! ZKO9 14 RO00027830 OPERATOR TRAINEE CASHIER EXCHANGE ARE ALLOWED WITHIN 7 DAYS WITH RECEIPT. STRICTLY NO CASH REFUND."



date: successful matches (462 available)


...

,image_id,address,ocr_status,ocr_text
0,X00016469612,"NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",success,"tan woon yann BOOK TA -K (TAMAN DAYA) SDN BHD PRON 7-W NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU, JOHOR. WAM MITA At Document Ho : TDO1167104 Date 25/12/2018 8:13:39 PM Cashier MANIS. Member CASH BILL CODE/DESC PRICE Dise AMOUIHT Quy RM RM 0556929040118 AF MODELLING CLAY KIDDY FISH 1PC + 9.00) 0,00 9.00 Total : 9,00 Rour ding Adjustment 0.00 Round. :d Total (RM): 9.00 Cash oy 40.00. CHANGE 00 GOODS SOLD ARE NOT RETURNAP EXCHANGEABLE THANK YOU PLEASE COME AGAIN t"
1,X00016469619,"27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JOHOR BAHRU, JOHOR.",success,"tan woon yann INDAH GIFT & HOME BECO 27, JALAN DEDAP 13, TANAN JOHOR JAYA, 81100 JOHOR BAHRU, JOHOR. Tel:07-3507405 Fax:07-35598160 RECEIPT 19/10/2018 20:49:59 #01 Cashier: CN Location/SP: 05 /0531 MB: 4026588 Room No: Of 050100025279 Desc/Iten Qty Price Ant/RH) ST-PRIVILEGE CARD/GD INDAH 88888 1 10.00 10.00 GF-TABLE LAMP/STITCH 62483 1 55.90 @PIsc 10.00% HTotal Qty 2 TOTAL AHT..+« ROUNDING ADI. CASH... CHANGE. « RH 10.00 Thank You ! Please Come Again ! Goods Sold Are Not Retur Healing In Wholesale And"
4,X00016469623,"LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR (TESCO PUTRA NILAI)",success,"tan woon yann . (MW) SON BHD 86467 |-D) OG1-A &@ 1Q51-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR (TESCO PUTRA NILAI) INVOICE~ KELAT AUTO FCO WASH & SHINE ES1000 11 WAdS-/2A - 12 9555916500193 exe 3.17 3.11 * ECO AUTO WASH &HAX EW-1000-11 12 1X 4.62 4.62 500084 TX 10.23 11.29 KLEENSO AJAIB 99 SERAI WANYI 9006 WOOO - 15 9555651400385 Le TAS 14S HANDKERCHIEF 7138682PCS PLi2Pdl 1-4 - 6/900 gO090822 1X 4.50 4.50 Item(s) : 6 Qty(s) : 5 Tota] RM 30,91 ROUNDING ADJUSTMENT] RM 0.01 TOTAL ROUNDED RM 30.90 CASH RM 51.00 CHANGE RM 20.10 + + 18-11-18 13:58 SHOI 2153 12 RONG002902 UPERATOR TRAINEE CASHIER EXCHANGE ARE ALLOWED WITHIN 7 DAYS WITH RECEIPT. STRICTLY NO CASH REFUND,"



total: successful matches (478 available)


,image_id,total,ocr_status,ocr_text
0,X00016469612,9.00,success,"tan woon yann BOOK TA -K (TAMAN DAYA) SDN BHD PRON 7-W NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU, JOHOR. WAM MITA At Document Ho : TDO1167104 Date 25/12/2018 8:13:39 PM Cashier MANIS. Member CASH BILL CODE/DESC PRICE Dise AMOUIHT Quy RM RM 0556929040118 AF MODELLING CLAY KIDDY FISH 1PC + 9.00) 0,00 9.00 Total : 9,00 Rour ding Adjustment 0.00 Round. :d Total (RM): 9.00 Cash oy 40.00. CHANGE 00 GOODS SOLD ARE NOT RETURNAP EXCHANGEABLE THANK YOU PLEASE COME AGAIN t"
2,X00016469620,33.90,success,"tan woon yann MR O.1.¥. (JOHOR) SDN BHD (CO.REG :933109-x) LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG , 43300 SERI KEMBANGAN, SELANGOR (MR DIY TESCO TERBAU) ~INVOICE~ CHOPPING BOARO 35.5x25.5CM 803M# EZiOHDOS - 24 8970669 1X 19.00 19.00 AIR PRESSURE SPRAYER SX~575-1 1.5L HCO3-7 - 15 9066468 8.02 8.02 WAXCO WINDSHILED cLeaneR” 120ML WAI4-3A - 48 9557031100236 1X 3.02 3.02 BOPP TAPE 48MM#100M CLEAR FZ-04 - 36 6935818350846 1X 3.88 3,88 Item(s) : 4 TOTAL RM 33, 92 ROUNDING ADJUSTMED -RM 0.02 TOTAL ROUNDED RH 33,90 CASH RM 50.00 CHANGE RM 16.10 #2222222 +--+ ---- ed 12-01-19 21:13 SHO! ZKO9 14 RO00027830 OPERATOR TRAINEE CASHIER EXCHANGE ARE ALLOWED WITHIN 7 DAYS WITH RECEIPT. STRICTLY NO CASH REFUND."
3,X00016469622,80.90,success,"tan woon yann YONGFATT ENTERPRISE (4MO517726) NO 122.124. JALAN DEDAP 13 81100 JOHOR BAHRU TEL © 3888, GST ID 000849812504 SIMPLIFIED TAX INVOICE CASH Doc No 0031664 — Date 26/12/2018 USER Time 123100 sperson Ret fom Qty SiPrice Amount Tax £8318 1 8094 8091 SR ELEGANT SCH TR BAG 15 Total Qty 1 8094 Total Sales (Excluding GST) 6091 Discount 000 Total GST 0.00 Rounding 001 elusive of GST) = 80.90 CASH : 100.00 Change: 19.10 | AmtU{RM) Tax (Rm) | 80.91 0.00! Total 80.91 0.00| ARE NOT REFUNDABLE ONLY"



total: OCR mismatches (146 available)


,image_id,total,ocr_status,ocr_text
1,X00016469619,60.30,success,"tan woon yann INDAH GIFT & HOME BECO 27, JALAN DEDAP 13, TANAN JOHOR JAYA, 81100 JOHOR BAHRU, JOHOR. Tel:07-3507405 Fax:07-35598160 RECEIPT 19/10/2018 20:49:59 #01 Cashier: CN Location/SP: 05 /0531 MB: 4026588 Room No: Of 050100025279 Desc/Iten Qty Price Ant/RH) ST-PRIVILEGE CARD/GD INDAH 88888 1 10.00 10.00 GF-TABLE LAMP/STITCH 62483 1 55.90 @PIsc 10.00% HTotal Qty 2 TOTAL AHT..+« ROUNDING ADI. CASH... CHANGE. « RH 10.00 Thank You ! Please Come Again ! Goods Sold Are Not Retur Healing In Wholesale And"
5,X00016469669,31.00,success,"tan chay yee ABC HO TRADING No,2&4, JALAN HARMONT 3/2, TAMAN DESA HARMONT. 81100 JOHOR BAHRUJOHOR 07-355 2616 Cash Bill ; ~01-143008 Date —; 09/01/2019 8:01:11 PM Cashier: OL Description Qty = Price: Amount RM “T — ""2 15.50 Total Amount: Cash Received Change : *Goods Sold are non-refundabl ***Thank You*** 3 ).00 **""Please Come Again***"
8,X51005200938,112.45,success,"Axo dig: PERNIAGAAR ZHENG HUI JMOs2B956-V NO.89 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81760 JOHOR BAHRU. TEL: 07-386 7524 FAX: 07-386 3793. GST NO ; odan0889624 ‘SIMPLIFIED TAX INVOICE “GOGIANT ENGINEERING ( ° Recslpt# CS00082562 Salesperson! Date: 1202/2018 Cashier: USER Time:08:30°20 TR ih, tiem _. Sty _RSP__ Amount azag 6B 1 1600 18.6¢ SR: 11"" PROWESS H/DUTY SILICONE GUN G-D2 8586075109147 aA 3 7.00 21.05 SR: XTRASEAL RIV ACETIC SILICONE SA-107 Orda GBo1 1800 © 48.00 SR: CENTRAL PUNCH SET SPCS- 1/4"",5/32"".1/4"" 12625 ae 1 8000 saac ‘SR. 24"" MARKSMAN PIPE WRENCH. 2016 GA 3 070 24C SR: PVC WIRE TAPE 1PC X 7YD “TOT any (Excluded GST) Sub Total (RM) 706.10 Discourt (RM) 0,00 Total OST (RM) 637 Rounding (RM) : ~0.02 Total (Flay + 112.46 CASH: 112.48 Change (RM): 6.00 GST SUMMARY GOODS SOLD ARE NOT RETURNABLE. THANK YOU."


In [12]:
if OCR_PATH.is_file() and not coverage.empty:

    lowest_coverage = coverage['coverage_percent'].sort_values().index.tolist()

    print('Fields most affected by normalized substring misses: ' + ', '.join(lowest_coverage))

    print('Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.')

else:

    print('No field-level OCR conclusion is reported because data/processed/sroie_ocr.jsonl is unavailable.')

    print('Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.')

Fields most affected by normalized substring misses: address, company, date, total
Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.


## Interpretation and next experiments

Use the coverage table and examples above to identify the lowest-coverage fields. Typical failure patterns to inspect are character substitutions in company names, punctuation or line-layout damage in addresses, date separator/character confusion, and decimal/comma confusion in totals.

The next preprocessing experiments should be isolated and measured against this baseline: grayscale conversion and contrast normalization, light denoising, adaptive thresholding, deskewing, receipt cropping or border removal, and a small set of Tesseract page-segmentation modes. Re-run the same comparison after each experiment while preserving the original OCR JSONL for auditability.